# Inside Steam - Data Audit

## 1. Dataset Overview

This notebook performs an initial audit of the raw Steam dataset in order to understand its structure, data quality, available variables, and analytical potential before cleaning and preprocessing.
The raw March 2025 file contains 47 variables across approximately 95,000 Steam games.

In [3]:
import pandas as pd
import numpy as np

In [1]:
file_path = "../data/raw/games_march2025_full.csv"

In [4]:
columns = pd.read_csv(file_path, nrows=0).columns

print(f"Number of columns: {len(columns)}")
columns.tolist()

Number of columns: 47


['appid',
 'name',
 'release_date',
 'required_age',
 'price',
 'dlc_count',
 'detailed_description',
 'about_the_game',
 'short_description',
 'reviews',
 'header_image',
 'website',
 'support_url',
 'support_email',
 'windows',
 'mac',
 'linux',
 'metacritic_score',
 'metacritic_url',
 'achievements',
 'recommendations',
 'notes',
 'supported_languages',
 'full_audio_languages',
 'packages',
 'developers',
 'publishers',
 'categories',
 'genres',
 'screenshots',
 'movies',
 'user_score',
 'score_rank',
 'positive',
 'negative',
 'estimated_owners',
 'average_playtime_forever',
 'average_playtime_2weeks',
 'median_playtime_forever',
 'median_playtime_2weeks',
 'discount',
 'peak_ccu',
 'tags',
 'pct_pos_total',
 'num_reviews_total',
 'pct_pos_recent',
 'num_reviews_recent']

## 2. Dataset Structure

In [5]:
df_sample = pd.read_csv(
    file_path,
    nrows=5000,
    low_memory=False
)

df_sample.head()

,appid,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,average_playtime_2weeks,median_playtime_forever,median_playtime_2weeks,discount,peak_ccu,tags,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0,0.00,1,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",NaN,...,879,5174,350,0,1212356,"{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0,0.00,0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,NaN,...,0,0,0,0,616738,"{'Survival': 14838, 'Shooter': 12727, 'Battle ...",59,2513842,68,16720
2,570,Dota 2,2013-07-09,0,0.00,2,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",“A modern multiplayer masterpiece.” 9.5/10 – D...,...,1536,898,892,0,555977,"{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,17,0.00,0,"When a young street hustler, a retired bank ro...","When a young street hustler, a retired bank ro...",Grand Theft Auto V for PC offers players the o...,NaN,...,771,7101,74,0,117698,"{'Open World': 32644, 'Action': 23539, 'Multip...",87,1803832,92,17517
4,488824,Tom Clancy's Rainbow Six® Siege,2015-12-01,17,19.99,9,Edition Comparison Ultimate Edition The Tom Cl...,“One of the best first-person shooters ever ma...,"Tom Clancy's Rainbow Six® Siege is an elite, t...",NaN,...,0,0,0,0,0,"{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",84,1168404,76,13017


In [6]:
df_sample.shape

(5000, 47)

In [7]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 47 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   appid                     5000 non-null   int64  
 1   name                      5000 non-null   str    
 2   release_date              5000 non-null   str    
 3   required_age              5000 non-null   int64  
 4   price                     5000 non-null   float64
 5   dlc_count                 5000 non-null   int64  
 6   detailed_description      4421 non-null   str    
 7   about_the_game            4416 non-null   str    
 8   short_description         4436 non-null   str    
 9   reviews                   1584 non-null   str    
 10  header_image              5000 non-null   str    
 11  website                   3408 non-null   str    
 12  support_url               3228 non-null   str    
 13  support_email             3002 non-null   str    
 14  windows            

In [8]:
df_sample.dtypes.value_counts()

str        23
int64      19
bool        3
float64     2
Name: count, dtype: int64

In [9]:
audit_table = pd.DataFrame({
    "dtype": df_sample.dtypes,
    "missing_pct": (df_sample.isna().mean() * 100).round(2),
    "unique_values": df_sample.nunique()
})

audit_table

,dtype,missing_pct,unique_values
appid,int64,0.00,5000
name,str,0.00,4950
release_date,str,0.00,2515
required_age,int64,0.00,8
price,float64,0.00,153
dlc_count,int64,0.00,84
detailed_description,str,11.58,4368
about_the_game,str,11.68,4362
short_description,str,11.28,4364
reviews,str,68.32,1559


In [10]:
audit_table.sort_values("missing_pct", ascending=False)

,dtype,missing_pct,unique_values
score_rank,float64,99.94,3
notes,str,79.80,892
metacritic_url,str,69.96,1465
reviews,str,68.32,1559
support_email,str,39.96,2132
support_url,str,35.44,2168
website,str,31.84,3127
about_the_game,str,11.68,4362
detailed_description,str,11.58,4368
short_description,str,11.28,4364


## 3. Key Variables and Analytical Potential

In [11]:
key_columns = [
    "appid",
    "name",
    "release_date",
    "price",
    "discount",
    "developers",
    "publishers",
    "genres",
    "categories",
    "tags",
    "metacritic_score",
    "user_score",
    "positive",
    "negative",
    "recommendations",
    "estimated_owners",
    "average_playtime_forever",
    "median_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_2weeks",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent"
]

df_sample[key_columns].head(10)

,appid,name,release_date,price,discount,developers,publishers,genres,categories,tags,...,estimated_owners,average_playtime_forever,median_playtime_forever,average_playtime_2weeks,median_playtime_2weeks,peak_ccu,pct_pos_total,num_reviews_total,pct_pos_recent,num_reviews_recent
0,730,Counter-Strike 2,2012-08-21,0.00,0,['Valve'],['Valve'],"['Action', 'Free To Play']","['Multi-player', 'Cross-Platform Multiplayer',...","{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",...,100000000 - 200000000,33189,5174,879,350,1212356,86,8632939,82,96473
1,578080,PUBG: BATTLEGROUNDS,2017-12-21,0.00,0,['PUBG Corporation'],"['KRAFTON, Inc.']","['Action', 'Adventure', 'Massively Multiplayer...","['Multi-player', 'PvP', 'Online PvP', 'Stats',...","{'Survival': 14838, 'Shooter': 12727, 'Battle ...",...,50000000 - 100000000,0,0,0,0,616738,59,2513842,68,16720
2,570,Dota 2,2013-07-09,0.00,0,['Valve'],['Valve'],"['Action', 'Strategy', 'Free To Play']","['Multi-player', 'Co-op', 'Steam Trading Cards...","{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",...,200000000 - 500000000,43031,898,1536,892,555977,81,2452595,80,29366
3,271590,Grand Theft Auto V Legacy,2015-04-13,0.00,0,['Rockstar North'],['Rockstar Games'],"['Action', 'Adventure']","['Single-player', 'Multi-player', 'PvP', 'Onli...","{'Open World': 32644, 'Action': 23539, 'Multip...",...,50000000 - 100000000,19323,7101,771,74,117698,87,1803832,92,17517
4,488824,Tom Clancy's Rainbow Six® Siege,2015-12-01,19.99,0,['Ubisoft Montreal'],['Ubisoft'],['Action'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",...,0 - 20000,0,0,0,0,0,84,1168404,76,13017
5,488822,Tom Clancy's Rainbow Six® Siege,2015-12-01,19.99,0,['Ubisoft Montreal'],['Ubisoft'],['Action'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","{'FPS': 8078, 'Multiplayer': 6138, 'Tactical':...",...,0 - 20000,0,0,0,0,0,84,1168404,76,13017
6,488821,Tom Clancy's Rainbow Six® Siege,2015-12-01,19.99,0,['Ubisoft Montreal'],['Ubisoft'],['Action'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","{'FPS': 8078, 'Multiplayer': 6136, 'Tactical':...",...,0 - 20000,0,0,0,0,0,84,1168404,76,13017
7,488823,Tom Clancy's Rainbow Six® Siege,2015-12-01,19.99,0,['Ubisoft Montreal'],['Ubisoft'],['Action'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",...,0 - 20000,0,0,0,0,0,84,1168404,76,13017
8,359550,Tom Clancy's Rainbow Six® Siege,2015-12-01,3.99,80,['Ubisoft Montreal'],['Ubisoft'],['Action'],"['Single-player', 'Multi-player', 'PvP', 'Onli...","{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",...,20000000 - 50000000,14204,2434,682,306,89916,84,1168020,76,12608
9,440,Team Fortress 2,2007-10-10,0.00,0,['Valve'],['Valve'],"['Action', 'Free To Play']","['Multi-player', 'Cross-Platform Multiplayer',...","{'Free to Play': 62868, 'Hero Shooter': 61020,...",...,20000000 - 50000000,0,0,0,0,50817,89,1146642,93,8172


In [12]:
df_sample["estimated_owners"].value_counts()

estimated_owners
200000 - 500000          1272
500000 - 1000000          868
0 - 0                     848
100000 - 200000           574
1000000 - 2000000         530
2000000 - 5000000         359
50000 - 100000            168
0 - 20000                 156
5000000 - 10000000        105
20000 - 50000              44
10000000 - 20000000        41
20000000 - 50000000        23
50000000 - 100000000       10
100000000 - 200000000       1
200000000 - 500000000       1
Name: count, dtype: int64

In [13]:
df_sample[["genres", "tags", "developers", "publishers", "categories"]].head(10)

,genres,tags,developers,publishers,categories
0,"['Action', 'Free To Play']","{'FPS': 90857, 'Shooter': 65397, 'Multiplayer'...",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer',..."
1,"['Action', 'Adventure', 'Massively Multiplayer...","{'Survival': 14838, 'Shooter': 12727, 'Battle ...",['PUBG Corporation'],"['KRAFTON, Inc.']","['Multi-player', 'PvP', 'Online PvP', 'Stats',..."
2,"['Action', 'Strategy', 'Free To Play']","{'Free to Play': 59933, 'MOBA': 20158, 'Multip...",['Valve'],['Valve'],"['Multi-player', 'Co-op', 'Steam Trading Cards..."
3,"['Action', 'Adventure']","{'Open World': 32644, 'Action': 23539, 'Multip...",['Rockstar North'],['Rockstar Games'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
4,['Action'],"{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",['Ubisoft Montreal'],['Ubisoft'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
5,['Action'],"{'FPS': 8078, 'Multiplayer': 6138, 'Tactical':...",['Ubisoft Montreal'],['Ubisoft'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
6,['Action'],"{'FPS': 8078, 'Multiplayer': 6136, 'Tactical':...",['Ubisoft Montreal'],['Ubisoft'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
7,['Action'],"{'FPS': 8082, 'Multiplayer': 6139, 'Tactical':...",['Ubisoft Montreal'],['Ubisoft'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
8,['Action'],"{'FPS': 9831, 'PvP': 9162, 'e-sports': 9072, '...",['Ubisoft Montreal'],['Ubisoft'],"['Single-player', 'Multi-player', 'PvP', 'Onli..."
9,"['Action', 'Free To Play']","{'Free to Play': 62868, 'Hero Shooter': 61020,...",['Valve'],['Valve'],"['Multi-player', 'Cross-Platform Multiplayer',..."


In [14]:
zero_check_columns = [
    "metacritic_score",
    "user_score",
    "recommendations",
    "positive",
    "negative",
    "average_playtime_forever",
    "median_playtime_forever",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total"
]

zero_summary = pd.DataFrame({
    "zero_count": (df_sample[zero_check_columns] == 0).sum(),
    "zero_pct": ((df_sample[zero_check_columns] == 0).mean() * 100).round(2)
})

zero_summary.sort_values("zero_pct", ascending=False)

,zero_count,zero_pct
user_score,4997,99.94
metacritic_score,3498,69.96
average_playtime_forever,2495,49.90
median_playtime_forever,2495,49.90
recommendations,1157,23.14
peak_ccu,982,19.64
positive,919,18.38
negative,918,18.36
pct_pos_total,0,0.00
num_reviews_total,0,0.00


### Initial Audit Observations

- The dataset contains several multi-valued fields such as genres, categories, developers and publishers, currently stored as string representations of lists.
- The `tags` field contains a dictionary-like structure associating tags with popularity/vote counts.
- `estimated_owners` is stored as ownership ranges rather than as a numeric value and will require transformation before quantitative analysis.
- Several numeric variables use zero very frequently, suggesting that zero may sometimes represent unavailable information rather than a true measured value.
- `user_score` appears almost entirely unavailable in the initial sample and may have limited analytical value.
- `metacritic_score` is zero for a large proportion of the initial sample.
- Lifetime playtime is zero for approximately half of the initial sample, while `peak_ccu`, recommendations and review counts provide additional engagement/reach signals.
- `appid` should be used as the primary game identifier, as game names are not necessarily unique.
- These observations are based on the first 5,000 rows and must be validated on the full dataset before making cleaning decisions.

In [15]:
analysis_columns = [
    "appid",
    "name",
    "release_date",
    "price",
    "discount",
    "windows",
    "mac",
    "linux",
    "developers",
    "publishers",
    "genres",
    "metacritic_score",
    "user_score",
    "recommendations",
    "positive",
    "negative",
    "estimated_owners",
    "average_playtime_forever",
    "average_playtime_2weeks",
    "median_playtime_forever",
    "median_playtime_2weeks",
    "peak_ccu",
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent"
]

df_audit = pd.read_csv(
    file_path,
    usecols=analysis_columns,
    low_memory=False
)

df_audit.shape

(94948, 26)

In [16]:
full_audit = pd.DataFrame({
    "dtype": df_audit.dtypes,
    "missing_pct": (df_audit.isna().mean() * 100).round(2),
    "zero_pct": [
        round((df_audit[col] == 0).mean() * 100, 2)
        if pd.api.types.is_numeric_dtype(df_audit[col])
        else np.nan
        for col in df_audit.columns
    ],
    "unique_values": df_audit.nunique()
})

full_audit

,dtype,missing_pct,zero_pct,unique_values
appid,int64,0.0,0.00,94948
name,str,0.0,NaN,94191
release_date,str,0.0,NaN,4464
price,float64,0.0,20.45,616
windows,bool,0.0,0.03,2
mac,bool,0.0,81.60,2
linux,bool,0.0,86.67,2
metacritic_score,int64,0.0,96.23,69
recommendations,int64,0.0,82.31,4672
developers,str,0.0,NaN,56503


In [17]:
df_audit["user_score"].value_counts().sort_index()

user_score
0      94909
37         1
46         2
51         2
53         1
55         1
57         1
60         1
61         1
63         1
65         1
66         1
68         2
71         1
73         1
76         1
77         2
78         1
80         2
82         1
83         1
84         2
87         1
88         1
92         1
94         1
95         1
96         1
97         1
100        5
Name: count, dtype: int64

In [18]:
df_audit["metacritic_score"].value_counts().sort_index().head(15)

metacritic_score
0     91372
20        1
23        1
29        1
30        1
32        2
34        3
35        1
36        2
37        3
38        3
39        4
40        5
41        7
42        3
Name: count, dtype: int64

In [19]:
df_audit["pct_pos_total"].describe()

count    94948.000000
mean        44.630261
std         40.837047
min         -1.000000
25%         -1.000000
50%         58.000000
75%         84.000000
max        100.000000
Name: pct_pos_total, dtype: float64

In [20]:
df_audit["num_reviews_total"].describe()

count    9.494800e+04
mean     1.448044e+03
std      3.548141e+04
min     -1.000000e+00
25%     -1.000000e+00
50%      1.500000e+01
75%      8.000000e+01
max      8.632939e+06
Name: num_reviews_total, dtype: float64

### Full Dataset Audit – Key Findings

- The dataset contains 94,948 Steam applications.
- `appid` is unique for every record and will be used as the primary game identifier.
- Several variables use special numeric values instead of standard missing values.
- `user_score` has extremely limited coverage and is unlikely to be useful for the main analysis.
- `metacritic_score` is unavailable for most games and should be treated as a secondary analytical variable rather than a core KPI.
- Review-related variables use `-1` to indicate unavailable information, which will need to be converted during preprocessing.
- Zero values cannot be treated uniformly as missing data: for variables such as price, discount and platform availability, zero/False values have valid business meanings.
- Playtime and concurrent-player metrics have limited coverage and will require careful filtering or availability flags before being used for engagement analysis.
- `estimated_owners` is categorical and stored as ownership ranges rather than exact numeric counts.

In [22]:
review_columns = [
    "pct_pos_total",
    "num_reviews_total",
    "pct_pos_recent",
    "num_reviews_recent"
]

sentinel_summary = pd.DataFrame({
    "minus_one_count": (df_audit[review_columns] == -1).sum(),
    "minus_one_pct": ((df_audit[review_columns] == -1).mean() * 100).round(2),
    "min_value": df_audit[review_columns].min(),
    "max_value": df_audit[review_columns].max()
})

sentinel_summary

,minus_one_count,minus_one_pct,min_value,max_value
pct_pos_total,39575,41.68,-1,100
num_reviews_total,39575,41.68,-1,8632939
pct_pos_recent,87727,92.39,-1,100
num_reviews_recent,87727,92.39,-1,96473


In [23]:
(
    (df_audit["num_reviews_total"] == -1)
    ==
    (df_audit["pct_pos_total"] == -1)
).mean()

np.float64(1.0)

### Review Sentinel Values

The review-related columns (`pct_pos_total`, `num_reviews_total`, `pct_pos_recent`, `num_reviews_recent`) use `-1` as a sentinel value for unavailable review information rather than as a valid measurement.

The sentinel convention is internally consistent:
- `num_reviews_total == -1` always corresponds to `pct_pos_total == -1`
- the same logic applies to the recent review fields

During preprocessing, these `-1` values will be converted to missing values (`NaN`) so they are not treated as real observations in statistical analysis.

In [24]:
df_audit["price"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99]
)

count    94948.000000
mean         6.911444
std         13.071148
min          0.000000
25%          0.990000
50%          3.990000
75%          9.990000
90%         14.990000
95%         19.990000
99%         39.990000
max        999.980000
Name: price, dtype: float64

In [25]:
(df_audit["price"] == 0).sum()

np.int64(19420)

In [27]:
df_audit.nlargest(
    20,
    "price"
)[["appid", "name", "price", "developers", "publishers", "genres"]]

,appid,name,price,developers,publishers,genres
64185,2504210,The Leverage Game Business Edition,999.98,['A&S Inc.'],['A&S Inc.'],"['Indie', 'Simulation']"
86496,2499620,The Leverage Game,999.98,['A&S Inc.'],['A&S Inc.'],"['Indie', 'Simulation']"
69953,1200520,Ascent Free-Roaming VR Experience,999.00,['Fury Games'],['Fury Games'],['Action']
89994,3013840,True Love,500.00,['A Guy'],['Whoes heart broken'],"['Action', 'Adventure', 'Casual', 'Indie']"
77589,253670,Aartform Curvy 3D 3.0,299.90,['Aartform'],['Aartform'],['Animation & Modeling']
77921,502570,Houdini Indie,269.99,['SideFX'],['SideFX'],"['Animation & Modeling', 'Design & Illustratio..."
78188,438450,3DF Zephyr Lite Steam Edition,249.00,['3Dflow SRL'],['3Dflow SRL'],"['Animation & Modeling', 'Design & Illustratio..."
56794,1368670,Beach Volleyball Competition,219.00,['healingdrawing'],['healingdrawing'],"['Action', 'Sports']"
31440,1873990,3D PUZZLE - Old House,199.99,['PUZZLE Games'],['Hede'],"['Action', 'Adventure', 'Casual', 'Indie', 'Si..."
34099,1912700,3D PUZZLE - Wood House,199.99,['PUZZLE Games'],['Hede'],"['Action', 'Adventure', 'Casual', 'Indie', 'Si..."


In [28]:
for threshold in [50, 100, 200, 500]:
    count = (df_audit["price"] > threshold).sum()
    pct = count / len(df_audit) * 100
    
    print(f"Price > ${threshold}: {count:,} games ({pct:.3f}%)")

Price > $50: 585 games (0.616%)
Price > $100: 184 games (0.194%)
Price > $200: 8 games (0.008%)
Price > $500: 3 games (0.003%)


### Price Distribution

The price distribution is strongly right-skewed. The median game price is $3.99, while the mean is higher at $6.91 due to a small number of expensive titles.

Most games remain relatively inexpensive:
- 75% are priced at $9.99 or below
- 95% are priced at $19.99 or below
- 99% are priced at $39.99 or below

Extremely high prices are very rare:
- 585 games (0.616%) are priced above $50
- 184 games (0.194%) are priced above $100
- 8 games (0.008%) are priced above $200
- 3 games (0.003%) are priced above $500

Manual verification confirmed that at least some of the extreme values correspond to actual Steam prices rather than data errors. These observations will therefore be retained and handled carefully during analysis rather than automatically removed as outliers.

In [26]:
df_audit["estimated_owners"].value_counts()

estimated_owners
0 - 20000                59379
0 - 0                    13656
20000 - 50000             9692
50000 - 100000            4558
100000 - 200000           3009
200000 - 500000           2474
500000 - 1000000          1042
1000000 - 2000000          576
2000000 - 5000000          376
5000000 - 10000000         108
10000000 - 20000000         42
20000000 - 50000000         24
50000000 - 100000000        10
100000000 - 200000000        1
200000000 - 500000000        1
Name: count, dtype: int64

### Estimated Owners Distribution

The estimated ownership distribution is highly concentrated in the lower ranges.

Approximately 77% of the Steam catalogue has 20,000 estimated owners or fewer, while only a small fraction of games reaches large audiences:
- around 8% exceed 100,000 estimated owners
- around 1.2% exceed 1 million estimated owners
- less than 0.1% exceed 10 million estimated owners

This suggests a strong long-tail structure in the Steam market, where a very small number of titles achieve very large reach while the majority remain relatively niche.

In [29]:
coverage = pd.DataFrame({
    "metric": [
        "Review positivity",
        "Recent review positivity",
        "Metacritic score",
        "Average lifetime playtime",
        "Median lifetime playtime",
        "Peak CCU",
        "Recommendations"
    ],
    "usable_count": [
        (df_audit["pct_pos_total"] >= 0).sum(),
        (df_audit["pct_pos_recent"] >= 0).sum(),
        (df_audit["metacritic_score"] > 0).sum(),
        (df_audit["average_playtime_forever"] > 0).sum(),
        (df_audit["median_playtime_forever"] > 0).sum(),
        (df_audit["peak_ccu"] > 0).sum(),
        (df_audit["recommendations"] > 0).sum()
    ]
})

coverage["coverage_pct"] = (
    coverage["usable_count"] / len(df_audit) * 100
).round(2)

coverage.sort_values("coverage_pct", ascending=False)

,metric,usable_count,coverage_pct
0,Review positivity,55373,58.32
5,Peak CCU,18940,19.95
6,Recommendations,16792,17.69
3,Average lifetime playtime,8019,8.45
4,Median lifetime playtime,8019,8.45
1,Recent review positivity,7221,7.61
2,Metacritic score,3576,3.77


In [30]:
owners_usable = (df_audit["estimated_owners"] != "0 - 0").sum()

print(
    f"Estimated owners usable: {owners_usable:,} "
    f"({owners_usable / len(df_audit) * 100:.2f}%)"
)

Estimated owners usable: 81,292 (85.62%)


### Data Coverage

The analytical variables do not have uniform coverage across the Steam catalogue, so different analyses will rely on different subsets of games.

Key coverage levels include:
- Estimated ownership: 85.62%
- Overall review positivity: 58.32%
- Peak concurrent users (CCU): 19.95%
- Recommendations: 17.69%
- Lifetime playtime: 8.45%
- Recent review positivity: 7.61%
- Metacritic score: 3.77%

Estimated ownership and overall player reviews provide sufficiently broad coverage for the main market reach and player reception analyses.

Engagement metrics such as playtime and peak CCU have more limited coverage and will therefore be analyzed on clearly defined subsets rather than treated as representative of the entire Steam catalogue.

Metacritic scores will be used only for secondary critic-versus-player analyses due to their limited coverage.